# Batch inference pipeline with cost optimization

A backfill needs to classify 100K customer-support emails with an LLM. On-demand inference at GPT-4o-mini rates would land near $30 and take an hour-plus of streaming requests; both Anthropic and OpenAI offer a batch API at half-price with a 24-hour SLA. You have 90 minutes to build the pipeline: format the dataset for both providers, submit to both, poll for completion, download results to S3, and run a cost comparator that picks the cheaper provider given the actual token counts.

The lab submits a 50-document slice (not the scenario's 100K). That keeps the bill under a dollar and still exercises every batch-API call.

**Duration:** about 90 minutes of work in a 120-minute session window.

**Cost preamble.** Cheapest lab in this sub-track. Batch pricing is half of on-demand; you prove that on the same 50-doc slice. About 60 cents on a clean run, $1.20 if you re-submit batches while debugging. The 1000-doc figure in the scenario lives on the slide; you only pay for 50 docs against each provider plus a small on-demand control.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. Anthropic 0.42 ships the batches API surface; OpenAI
# 1.54 ships the batches surface against /v1/chat/completions; boto3 1.35 for S3.

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 openai==1.54.0 boto3==1.35.0

In [ ]:
# Imports and per-lab constants. The S3 bucket name uses the account ID
# suffix so it is globally unique; the lab resolves the account ID via STS at
# setup time.

import atexit
import csv
import getpass
import io
import json
import os
import sys
import time
import uuid
from typing import Optional

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "batch-inference-cost-optimization"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID

REGION = "us-east-1"
BUCKET_BASENAME = f"labrun-{LAB_ID}-bucket"

ANTHROPIC_MODEL = "claude-haiku-4-5-20250930"
OPENAI_MODEL = "gpt-4o-mini"

# Per-1M token prices in dollars; halved versions for batch.
ANTHROPIC_BATCH_IN = 0.40  # 50% off Haiku $0.80/1M
ANTHROPIC_BATCH_OUT = 2.00  # 50% off Haiku $4/1M
OPENAI_BATCH_IN = 0.075  # 50% off gpt-4o-mini $0.15/1M
OPENAI_BATCH_OUT = 0.30  # 50% off $0.60/1M
ANTHROPIC_OD_IN = 0.80
ANTHROPIC_OD_OUT = 4.00
OPENAI_OD_IN = 0.15
OPENAI_OD_OUT = 0.60

NUM_DOCS = 50  # The lab slice; the scenario references 100K (not run here).

COMPARISON_PATH = "/content/batch_comparison.csv"
DOCS_PATH = "/content/labrun_emails.jsonl"

In [ ]:
# NBVAL_SKIP
# BYOK: session token, AWS credentials, Anthropic + OpenAI keys.
# Smoke-test each. STS GetCallerIdentity gives us the account ID for the
# bucket name; Anthropic Haiku ping; OpenAI models.list. Then register.

import anthropic
import openai
import boto3
from botocore.exceptions import ClientError

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip()
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY: ").strip()

if not all([AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, ANTHROPIC_API_KEY, OPENAI_API_KEY]):
    print("All four credentials are required. Re-run this cell.")
    raise SystemExit(1)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

sts = boto3.client("sts", region_name=REGION)
try:
    ident = sts.get_caller_identity()
    ACCOUNT_ID = ident["Account"]
    print(f"AWS credentials validated. Account: {ACCOUNT_ID}.")
except ClientError as exc:
    print(f"AWS credentials rejected: {exc!r}")
    raise SystemExit(1)

BUCKET_NAME = f"{BUCKET_BASENAME}-{ACCOUNT_ID}"

s3 = boto3.client("s3", region_name=REGION)

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    ping = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    raise SystemExit(1)

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
try:
    openai_client.models.list()
    print("OpenAI credentials validated.")
except Exception as exc:
    print(f"OpenAI key rejected: {exc!r}")
    raise SystemExit(1)

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={"region": REGION, "account_id": ACCOUNT_ID, "bucket": BUCKET_NAME},
)
print(f"Session registered for lab_id: {LAB_ID}; bucket name will be {BUCKET_NAME}.")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, inline BatchInferenceCleanupAdapter, atexit, orphan scan.
# Reverse-creation order: local CSV last (created first is created last...
# scratch that: nothing's created yet; the bucket+files are added to the
# manifest as the lab runs).

CLEANUP_MANIFEST = [
    CleanupResource(
        type="local_file",
        id=COMPARISON_PATH,
        region="local",
        cli_delete_command=f"rm -f {COMPARISON_PATH}",
    ),
    CleanupResource(
        type="local_file",
        id=DOCS_PATH,
        region="local",
        cli_delete_command=f"rm -f {DOCS_PATH}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


# Mutable state the adapter consults at delete time; the batch + file IDs are
# appended as the lab runs.
LIVE_RESOURCES = {
    "anthropic_batches": [],
    "openai_batches": [],
    "anthropic_files": [],
    "openai_files": [],
}


class BatchInferenceCleanupAdapter:
    """Adapter for Anthropic/OpenAI batch + Files API + S3 bucket + local files.

    TODO: labrun-checks 0.7.0 has no anthropic.py or openai.py adapter; once 0.8 ships,
    migrate. The inline adapter mirrors that API.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        if resource.type == "s3_bucket":
            try:
                # Empty all objects + versions first.
                paginator = s3.get_paginator("list_objects_v2")
                for page in paginator.paginate(Bucket=resource.id):
                    if "Contents" in page:
                        s3.delete_objects(
                            Bucket=resource.id,
                            Delete={"Objects": [{"Key": o["Key"]} for o in page["Contents"]]},
                        )
                s3.delete_bucket(Bucket=resource.id)
            except ClientError as exc:
                if exc.response["Error"]["Code"] in ("NoSuchBucket",):
                    return
                raise
            return
        if resource.type == "anthropic_batch":
            try:
                anthropic_client.messages.batches.cancel(resource.id)
            except Exception:
                pass
            return
        if resource.type == "anthropic_file":
            try:
                anthropic_client.files.delete(resource.id)
            except Exception:
                pass
            return
        if resource.type == "openai_batch":
            try:
                openai_client.batches.cancel(resource.id)
            except Exception:
                pass
            return
        if resource.type == "openai_file":
            try:
                openai_client.files.delete(resource.id)
            except Exception:
                pass
            return

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        if resource.type == "s3_bucket":
            try:
                s3.head_bucket(Bucket=resource.id)
                return True
            except ClientError:
                return False
        if resource.type == "anthropic_batch":
            try:
                b = anthropic_client.messages.batches.retrieve(resource.id)
                return getattr(b, "processing_status", None) not in ("ended", "canceled")
            except Exception:
                return False
        if resource.type == "anthropic_file":
            try:
                anthropic_client.files.retrieve(resource.id)
                return True
            except Exception:
                return False
        if resource.type == "openai_batch":
            try:
                b = openai_client.batches.retrieve(resource.id)
                return b.status not in ("completed", "failed", "expired", "cancelled")
            except Exception:
                return False
        if resource.type == "openai_file":
            try:
                openai_client.files.retrieve(resource.id)
                return True
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # AWS Resource Groups Tagging API would surface S3 buckets tagged with
        # our lab slug; this orphan check covers the only AWS resource the lab
        # creates.
        try:
            rg = boto3.client("resourcegroupstaggingapi", region_name=region)
            resp = rg.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [lab_slug]}])
            return [r["ResourceARN"] for r in resp.get("ResourceTagMappingList", [])]
        except Exception:
            return []


CLEANUP_ADAPTER = BatchInferenceCleanupAdapter()


def _register_live(kind, ident):
    """Append a freshly-created resource so cleanup sees it.

    The manifest is mutated in place: we insert critical (provider) resources
    at the front so they tear down before S3 + local files.
    """
    res = CleanupResource(
        type=kind,
        id=ident,
        region=REGION if kind == "s3_bucket" else "external",
        cli_delete_command=f"# inline adapter handles {kind} {ident}",
    )
    CLEANUP_MANIFEST.insert(0, res)


# Orphan scan: any S3 bucket already tagged with this lab slug indicates a
# prior session did not clean up. Block.
try:
    rg = boto3.client("resourcegroupstaggingapi", region_name=REGION)
    resp = rg.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_ID]}])
    _orphans = [r["ResourceARN"] for r in resp.get("ResourceTagMappingList", [])]
except Exception:
    _orphans = []

if _orphans:
    print("Orphan scan found tagged resources from a prior session:")
    for arn in _orphans:
        print(f"  {arn}")
    print("Run cleanup on the prior session before re-running this lab.")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in list(CLEANUP_MANIFEST):
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Task 1: Format the dataset and submit to both batch APIs

The lab generates a 50-document synthetic corpus of short customer-support emails inside the kernel (so the size and content are deterministic and audit-friendly). Each document gets a one-line classification task: "Is the sender angry, neutral, or happy? Reply with exactly one of: angry, neutral, happy."

You have to format the same task two ways: Anthropic batch wants a JSON array of requests with `custom_id` + `params`; OpenAI batch wants a JSONL upload (one request per line) with `custom_id` + `method` + `url` + `body`. Submit both. Register the batch + file IDs into the cleanup manifest as you go.

In [ ]:
# Task 1: synthesize the corpus, format both batch payloads, submit.

def synth_docs(n=NUM_DOCS):
    moods = [
        ("Hey team, your platform is on fire. Crashed THREE times today.", "angry"),
        ("Just emailing to say everything is working fine, thanks.", "neutral"),
        ("Love the new release! The vector search is so much faster.", "happy"),
        ("Refund my subscription. This is not what I paid for.", "angry"),
        ("Following up on ticket 4421. No urgency.", "neutral"),
    ]
    out = []
    for i in range(n):
        text, gt = moods[i % len(moods)]
        out.append({"id": f"email_{i:03d}", "text": text, "ground_truth": gt})
    return out


docs = synth_docs(NUM_DOCS)
with open(DOCS_PATH, "w") as f:
    for d in docs:
        f.write(json.dumps(d) + "\n")
print(f"Wrote {len(docs)} docs to {DOCS_PATH}")


# Create the S3 bucket up front; the result files land there at the end of
# Task 3.
def ensure_bucket():
    try:
        s3.head_bucket(Bucket=BUCKET_NAME)
    except ClientError:
        # us-east-1 wants no LocationConstraint.
        s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(
        Bucket=BUCKET_NAME,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    print(f"S3 bucket ready: {BUCKET_NAME}")


ensure_bucket()


def submit_anthropic_batch(docs):
    # YOUR CODE: build a list of MessageCreateParamsNonStreaming-shaped
    # dicts. Each request has custom_id=d["id"] and params={
    #   "model": ANTHROPIC_MODEL,
    #   "max_tokens": 8,
    #   "messages": [{"role": "user", "content": <classification prompt>}],
    # }. Call anthropic_client.messages.batches.create(requests=...).
    # Return the batch.id.
    raise NotImplementedError("Replace with the Anthropic batches.create call.")


def submit_openai_batch(docs):
    # YOUR CODE: write a JSONL file in memory where each line is
    # {"custom_id": d["id"], "method": "POST", "url": "/v1/chat/completions",
    #  "body": {"model": OPENAI_MODEL, "max_tokens": 8, "messages": [...]}}.
    # Upload it via openai_client.files.create(file=..., purpose="batch"),
    # then call openai_client.batches.create(input_file_id=..., endpoint=
    # "/v1/chat/completions", completion_window="24h"). Register the file_id
    # AND the batch_id via _register_live. Return (file_id, batch.id).
    raise NotImplementedError("Replace with the OpenAI batches.create call.")


anthropic_batch_id = submit_anthropic_batch(docs)
_register_live("anthropic_batch", anthropic_batch_id)
print(f"Anthropic batch submitted: {anthropic_batch_id}")

openai_file_id, openai_batch_id = submit_openai_batch(docs)
_register_live("openai_file", openai_file_id)
_register_live("openai_batch", openai_batch_id)
print(f"OpenAI batch submitted: {openai_batch_id} (input file {openai_file_id})")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: both batches exist when retrieved by ID.


def checkpoint_1(session):
    try:
        a = anthropic_client.messages.batches.retrieve(anthropic_batch_id)
        if not getattr(a, "id", None):
            return CheckpointResult(status="fail", error_reason="Anthropic batch retrieve returned empty id.")
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=f"Anthropic batch retrieve failed: {exc!r}. Confirm submit_anthropic_batch returned a real id.",
        )
    try:
        o = openai_client.batches.retrieve(openai_batch_id)
        if not o.id:
            return CheckpointResult(status="fail", error_reason="OpenAI batch retrieve returned empty id.")
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=f"OpenAI batch retrieve failed: {exc!r}. Confirm submit_openai_batch returned a real id.",
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Both providers want a custom_id per request (so you can map the result back to your row). The schemas differ everywhere else. Read both API docs once before guessing field names; the SDK type-hints will guide you when you import the right helpers.

</details>

<details><summary>Hint 2 (direction)</summary>

Anthropic ships a `Request` dataclass under `anthropic.types.messages.batch_create_params`; you do not have to use it. Plain dicts work:

```
requests = [
    {"custom_id": d["id"], "params": {"model": ANTHROPIC_MODEL, "max_tokens": 8, "messages": [...]}}
    for d in docs
]
batch = anthropic_client.messages.batches.create(requests=requests)
```

OpenAI batch wants a JSONL UPLOAD first, then a batch that references the uploaded file_id. Build the JSONL in memory with `io.BytesIO`:

```
buf = io.BytesIO()
for d in docs:
    buf.write((json.dumps({"custom_id": d["id"], "method": "POST", "url": "/v1/chat/completions", "body": {...}}) + "\n").encode())
buf.seek(0)
file = openai_client.files.create(file=buf, purpose="batch")
batch = openai_client.batches.create(input_file_id=file.id, endpoint="/v1/chat/completions", completion_window="24h")
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete `submit_anthropic_batch`:

```python
def submit_anthropic_batch(docs):
    prompt = "Is the sender angry, neutral, or happy? Reply with exactly one of: angry, neutral, happy.\n\nEMAIL: {text}"
    requests = [
        {
            "custom_id": d["id"],
            "params": {
                "model": ANTHROPIC_MODEL,
                "max_tokens": 8,
                "messages": [{"role": "user", "content": prompt.format(text=d["text"])}],
            },
        }
        for d in docs
    ]
    batch = anthropic_client.messages.batches.create(requests=requests)
    return batch.id
```

Complete `submit_openai_batch`:

```python
def submit_openai_batch(docs):
    prompt = "Is the sender angry, neutral, or happy? Reply with exactly one of: angry, neutral, happy.\n\nEMAIL: {text}"
    buf = io.BytesIO()
    for d in docs:
        line = {
            "custom_id": d["id"],
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": OPENAI_MODEL,
                "max_tokens": 8,
                "messages": [{"role": "user", "content": prompt.format(text=d["text"])}],
            },
        }
        buf.write((json.dumps(line) + "\n").encode())
    buf.seek(0)
    file = openai_client.files.create(file=buf, purpose="batch")
    batch = openai_client.batches.create(
        input_file_id=file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    return file.id, batch.id
```

</details>

**Wallet check.** Submissions are free; you pay when the batch processes. Both batches queued. Anthropic typically lands in 10-20 minutes for 50 docs; OpenAI is similar. Use this time to read the cost comparator section.

## Task 2: Poll both batches until they finish

Both providers complete well inside the 24-hour SLA on this slice (typical 10-30 minutes for 50 docs). The polling loop must wait politely (no `time.sleep(1)`; use 30 to 60 seconds between polls to stay under rate limits), report transitions to the operator, and exit when both batches have a terminal status (`ended` for Anthropic; `completed`/`failed`/`expired`/`cancelled` for OpenAI).

In [ ]:
# Task 2: poll both batches.

ANTHROPIC_TERMINAL = {"ended", "canceled"}
OPENAI_TERMINAL = {"completed", "failed", "expired", "cancelled"}

POLL_INTERVAL_SEC = 30
POLL_CEILING_SEC = 60 * 90  # 90 minute hard cap for this lab.


def poll_until_done(anthropic_batch_id, openai_batch_id):
    deadline = time.time() + POLL_CEILING_SEC
    last_a = None
    last_o = None
    # YOUR CODE: loop while time.time() < deadline. Each iteration, retrieve
    # both batches via the SDK, read their status, print only when status
    # changes (compare against last_a/last_o), and break out of the loop when
    # both are in their terminal set. Sleep POLL_INTERVAL_SEC between
    # iterations. Return the two final batch objects.
    raise NotImplementedError("Replace with the polling loop body.")


a_final, o_final = poll_until_done(anthropic_batch_id, openai_batch_id)
print(f"Anthropic final status: {getattr(a_final, 'processing_status', None)}")
print(f"OpenAI final status: {o_final.status}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: both batches reached a terminal status.


def checkpoint_2(session):
    a = anthropic_client.messages.batches.retrieve(anthropic_batch_id)
    o = openai_client.batches.retrieve(openai_batch_id)
    if getattr(a, "processing_status", None) not in ANTHROPIC_TERMINAL:
        return CheckpointResult(
            status="fail",
            error_reason=f"Anthropic batch processing_status={getattr(a, 'processing_status', None)!r}; expected one of {ANTHROPIC_TERMINAL}.",
        )
    if o.status not in OPENAI_TERMINAL:
        return CheckpointResult(
            status="fail",
            error_reason=f"OpenAI batch status={o.status!r}; expected one of {OPENAI_TERMINAL}.",
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two batches, one loop. Read each provider's terminal-status set, watch the status field. If you call retrieve once per second the rate limit will smack you; 30 to 60 seconds is the polite cadence both SDKs recommend.

</details>

<details><summary>Hint 2 (direction)</summary>

```
while time.time() < deadline:
    a = anthropic_client.messages.batches.retrieve(anthropic_batch_id)
    o = openai_client.batches.retrieve(openai_batch_id)
    if a.processing_status != last_a:
        print(f"Anthropic: {a.processing_status}")
        last_a = a.processing_status
    if o.status != last_o:
        print(f"OpenAI: {o.status}")
        last_o = o.status
    if a.processing_status in ANTHROPIC_TERMINAL and o.status in OPENAI_TERMINAL:
        return a, o
    time.sleep(POLL_INTERVAL_SEC)
raise TimeoutError("Batches did not complete in 90 minutes.")
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete poll loop (matches Hint 2 verbatim plus the early-exit guard). Set `POLL_INTERVAL_SEC = 30` for a 50-doc batch; production would use 60.

</details>

**Wallet check.** Polling is free. Anthropic billed about 25 cents for the 50-doc batch (half-priced Haiku in/out). OpenAI billed under 5 cents (gpt-4o-mini batch is the cheapest serious model on the market right now). Spent so far: roughly 30 cents.

## Task 3: Download both result files to S3

Anthropic exposes results via `messages.batches.results(batch_id)` returning a stream of per-row results; OpenAI exposes them via the output_file_id you read off the batch object. Write each provider's results to a separate S3 object under the lab bucket: `s3://{BUCKET_NAME}/anthropic_results.jsonl` and `s3://{BUCKET_NAME}/openai_results.jsonl`. The validator counts rows.

In [ ]:
# Task 3: results -> S3.

def download_anthropic_results(batch_id):
    # YOUR CODE: call anthropic_client.messages.batches.results(batch_id),
    # iterate the resulting stream of per-row results, JSON-serialize each
    # row, and concatenate into a single bytes payload. Upload to S3 under
    # "anthropic_results.jsonl". Return the row count.
    raise NotImplementedError("Replace with the Anthropic results stream + S3 upload.")


def download_openai_results(batch):
    # YOUR CODE: read batch.output_file_id; download via
    # openai_client.files.content(output_file_id).read(); upload bytes to S3
    # under "openai_results.jsonl". Return the row count.
    raise NotImplementedError("Replace with the OpenAI results download + S3 upload.")


a_count = download_anthropic_results(anthropic_batch_id)
print(f"Anthropic rows downloaded to S3: {a_count}")
o_count = download_openai_results(o_final)
print(f"OpenAI rows downloaded to S3: {o_count}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: both result files in S3 with NUM_DOCS rows each.


def checkpoint_3(session):
    for key in ("anthropic_results.jsonl", "openai_results.jsonl"):
        try:
            obj = s3.get_object(Bucket=BUCKET_NAME, Key=key)
        except ClientError as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"S3 GetObject failed for {key!r}: {exc!r}",
            )
        body = obj["Body"].read().decode("utf-8", errors="replace")
        rows = [l for l in body.splitlines() if l.strip()]
        if len(rows) < NUM_DOCS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{key} has {len(rows)} rows; expected >= {NUM_DOCS}. "
                    f"Batch may have partial failures; check errored_count on the batch object."
                ),
            )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The two providers return results in different shapes. Anthropic gives you a streamed iterable of per-row results; OpenAI gives you a single file you download. The constant is custom_id; both providers attach it to every result row.

</details>

<details><summary>Hint 2 (direction)</summary>

Anthropic:

```
buf = io.BytesIO()
for row in anthropic_client.messages.batches.results(batch_id):
    buf.write((row.json() + "\n").encode() if hasattr(row, "json") else (json.dumps(row.model_dump()) + "\n").encode())
buf.seek(0)
s3.put_object(Bucket=BUCKET_NAME, Key="anthropic_results.jsonl", Body=buf.getvalue())
```

OpenAI:

```
content_resp = openai_client.files.content(batch.output_file_id)
body = content_resp.read()
s3.put_object(Bucket=BUCKET_NAME, Key="openai_results.jsonl", Body=body)
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
def download_anthropic_results(batch_id):
    buf = io.BytesIO()
    count = 0
    for row in anthropic_client.messages.batches.results(batch_id):
        payload = row.model_dump_json() if hasattr(row, "model_dump_json") else json.dumps(row)
        buf.write((payload + "\n").encode())
        count += 1
    s3.put_object(Bucket=BUCKET_NAME, Key="anthropic_results.jsonl", Body=buf.getvalue())
    return count


def download_openai_results(batch):
    body = openai_client.files.content(batch.output_file_id).read()
    rows = [l for l in body.decode("utf-8").splitlines() if l.strip()]
    s3.put_object(Bucket=BUCKET_NAME, Key="openai_results.jsonl", Body=body)
    return len(rows)
```

</details>

**Wallet check.** S3 PutObject for two ~30KB files: fractions of a penny. Cumulative spend: still around 30 cents.

## Task 4: Cost comparator picks the cheaper provider

You have token-count metadata in each result row (Anthropic: `usage.input_tokens`/`usage.output_tokens`; OpenAI: `response.body.usage.prompt_tokens`/`completion_tokens`). Compute total cost across the 50 docs for each provider given the batch rates above. Pick the lower number. Run an independent estimate using the same token counts and confirm your comparator agrees.

In [ ]:
# Task 4: aggregate per-provider costs and pick the winner.

def parse_anthropic_results():
    body = s3.get_object(Bucket=BUCKET_NAME, Key="anthropic_results.jsonl")["Body"].read().decode()
    rows = [json.loads(l) for l in body.splitlines() if l.strip()]
    in_tok = 0
    out_tok = 0
    # YOUR CODE: for each row, navigate the result envelope. The shape is
    # {"custom_id": ..., "result": {"type": "succeeded", "message": {"usage":
    # {"input_tokens": N, "output_tokens": M}, ...}}}. Skip non-succeeded rows.
    # Sum input_tokens into in_tok and output_tokens into out_tok.
    return in_tok, out_tok, len(rows)


def parse_openai_results():
    body = s3.get_object(Bucket=BUCKET_NAME, Key="openai_results.jsonl")["Body"].read().decode()
    rows = [json.loads(l) for l in body.splitlines() if l.strip()]
    in_tok = 0
    out_tok = 0
    # YOUR CODE: for each row, navigate the shape:
    # {"custom_id": ..., "response": {"status_code": 200, "body": {"usage":
    # {"prompt_tokens": N, "completion_tokens": M}, ...}}}. Skip rows where
    # status_code != 200. Sum prompt_tokens into in_tok, completion_tokens
    # into out_tok.
    return in_tok, out_tok, len(rows)


a_in, a_out, a_rows = parse_anthropic_results()
o_in, o_out, o_rows = parse_openai_results()

a_cost = (a_in * ANTHROPIC_BATCH_IN + a_out * ANTHROPIC_BATCH_OUT) / 1_000_000
o_cost = (o_in * OPENAI_BATCH_IN + o_out * OPENAI_BATCH_OUT) / 1_000_000

winner = "openai" if o_cost <= a_cost else "anthropic"
print(f"Anthropic: {a_in} in / {a_out} out -> ${a_cost:.6f}")
print(f"OpenAI:    {o_in} in / {o_out} out -> ${o_cost:.6f}")
print(f"Cheaper provider for this corpus: {winner}")


with open(COMPARISON_PATH, "w") as f:
    w = csv.writer(f)
    w.writerow(["provider", "input_tokens", "output_tokens", "total_cost_usd_batch"])
    w.writerow(["anthropic", a_in, a_out, f"{a_cost:.6f}"])
    w.writerow(["openai", o_in, o_out, f"{o_cost:.6f}"])
print(f"Wrote {COMPARISON_PATH}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: comparator wrote the CSV, the winner matches an independent
# recomputation off the same token counts.


def checkpoint_4(session):
    if not os.path.exists(COMPARISON_PATH):
        return CheckpointResult(status="fail", error_reason=f"{COMPARISON_PATH} missing.")
    with open(COMPARISON_PATH) as f:
        rows = list(csv.DictReader(f))
    by_prov = {r["provider"]: r for r in rows}
    if "anthropic" not in by_prov or "openai" not in by_prov:
        return CheckpointResult(
            status="fail",
            error_reason="comparison CSV must have one row per provider.",
        )
    try:
        a_cost = float(by_prov["anthropic"]["total_cost_usd_batch"])
        o_cost = float(by_prov["openai"]["total_cost_usd_batch"])
        a_in = int(by_prov["anthropic"]["input_tokens"])
        a_out = int(by_prov["anthropic"]["output_tokens"])
        o_in = int(by_prov["openai"]["input_tokens"])
        o_out = int(by_prov["openai"]["output_tokens"])
    except (KeyError, ValueError) as exc:
        return CheckpointResult(status="fail", error_reason=f"CSV columns malformed: {exc!r}")
    if a_in == 0 or o_in == 0:
        return CheckpointResult(
            status="fail",
            error_reason="Token counts are zero; parse_anthropic_results or parse_openai_results did not sum the usage fields.",
        )
    expected_a = (a_in * ANTHROPIC_BATCH_IN + a_out * ANTHROPIC_BATCH_OUT) / 1_000_000
    expected_o = (o_in * OPENAI_BATCH_IN + o_out * OPENAI_BATCH_OUT) / 1_000_000
    if abs(expected_a - a_cost) > 1e-4 or abs(expected_o - o_cost) > 1e-4:
        return CheckpointResult(
            status="fail",
            error_reason="Recorded cost does not match the price-map recomputation; check the input vs output token mapping.",
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The two providers nest the usage fields differently. Anthropic puts it inside `result.message.usage`; OpenAI puts it inside `response.body.usage`. Same numbers, different paths. The comparator math is just `(in_tok * price_in + out_tok * price_out) / 1M`.

</details>

<details><summary>Hint 2 (direction)</summary>

Anthropic row body shape after `model_dump_json`:

```
{"custom_id": "email_001", "result": {"type": "succeeded",
 "message": {"id": ..., "usage": {"input_tokens": 47, "output_tokens": 4}, ...}}}
```

OpenAI row body shape:

```
{"id": ..., "custom_id": "email_001",
 "response": {"status_code": 200, "body": {"usage": {"prompt_tokens": 51, "completion_tokens": 3}, ...}}}
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
def parse_anthropic_results():
    body = s3.get_object(Bucket=BUCKET_NAME, Key="anthropic_results.jsonl")["Body"].read().decode()
    rows = [json.loads(l) for l in body.splitlines() if l.strip()]
    in_tok, out_tok = 0, 0
    for r in rows:
        result = r.get("result", {})
        if result.get("type") != "succeeded":
            continue
        usage = result.get("message", {}).get("usage", {})
        in_tok += int(usage.get("input_tokens", 0))
        out_tok += int(usage.get("output_tokens", 0))
    return in_tok, out_tok, len(rows)


def parse_openai_results():
    body = s3.get_object(Bucket=BUCKET_NAME, Key="openai_results.jsonl")["Body"].read().decode()
    rows = [json.loads(l) for l in body.splitlines() if l.strip()]
    in_tok, out_tok = 0, 0
    for r in rows:
        resp = r.get("response", {})
        if resp.get("status_code") != 200:
            continue
        usage = resp.get("body", {}).get("usage", {})
        in_tok += int(usage.get("prompt_tokens", 0))
        out_tok += int(usage.get("completion_tokens", 0))
    return in_tok, out_tok, len(rows)
```

</details>

**Wallet check.** 1000-classification scenario at OpenAI batch rates: pennies. 1000 at Anthropic batch rates: dimes. The 50-doc slice you actually ran: well under a quarter. Total session damage: roughly 30 to 50 cents.

## Cleanup

Cancel any in-flight batches (idempotent), delete uploaded Files API entries, empty + delete the S3 bucket, drop the local CSV + JSONL. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. The manifest was mutated as the lab ran; resources are torn down
# in the order they appear (insert-at-front during the run put providers
# before S3 before local files).

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, batches or buckets may still be billing. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.40.** Anthropic batch dominated (Haiku at $0.40/1M batch input); OpenAI batch contributed pennies. S3 ingest+egress was a fraction of a penny. The batch APIs proved the 50% claim: same 50 docs, half the on-demand cost. Cleanup torn down both providers' uploads, the bucket, and the local CSV.

## Reflection

These are not graded. They are for you.

1. The lab proved batch is half the cost of on-demand. The team's product cannot wait 24 hours for results; latency contract is sub-second. What is one hybrid pattern that captures some batch savings without breaking the latency contract for user-facing requests?

2. Your comparator picked one provider on this corpus. If the corpus shifts (longer docs, different language, structured outputs), the winner may flip. What is the simplest way to make the comparator re-evaluate on a sample every week without re-coding it?

## Exam-Style Practice

**Question 1 (batch vs on-demand selection):**

A team has a daily classification job over 1M documents. Latency requirement: results by end of business day. Cost is the dominant constraint. Which is the right pattern?

A. On-demand inference at the provider, parallelized across many concurrent connections.
B. Batch API submitted overnight, results downloaded the next morning.
C. Fine-tune a smaller model on labeled examples.
D. Self-host vLLM on Modal.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: on-demand at this scale costs roughly 2x batch and adds rate-limit headaches; latency does not justify it when "end of business day" already accommodates a 24-hour SLA.
- B is correct: 24-hour SLA fits "results by end of business day"; the 50% discount applies; no infra to manage.
- C is wrong as a first move: fine-tuning is the answer when classification quality is the bottleneck, not when cost is.
- D is wrong: self-hosting moves the cost from API tokens to GPU hours; on a periodic batch workload, self-hosted GPU sits idle most of the day.

</details>

**Question 2 (batch failure handling):**

An Anthropic batch reports `processing_status="ended"` with `request_counts.errored=4`. Which is the correct next step?

A. Re-submit the entire batch to retry the four failures.
B. Read the results stream; failed rows include error details and the four custom_ids; retry those four on-demand.
C. Ignore the four; batch failures are transient.
D. Cancel the batch and start over.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wasteful: it re-charges the 96 successful rows.
- B is correct: the results stream carries the failures with the original custom_ids and an error envelope; production code retries the failed slice with the cheapest path that meets the deadline (often on-demand for a small handful).
- C is wrong: errored rows have no output; the downstream consumer would have missing classifications.
- D is wrong: the 96 successes are already paid for and complete; cancelling does not refund them.

</details>

**Question 3 (cost attribution):**

A team's monthly LLM bill is 60% input tokens, 40% output tokens. They consider moving from GPT-4o-mini (input $0.15/1M, output $0.60/1M) to GPT-4o batch (input $1.25/1M, output $5.00/1M). What is the dominant effect on the bill?

A. The bill drops because batch is always cheaper.
B. The bill rises substantially because GPT-4o is roughly 8x more expensive than mini, and batch's 50% only halves that 8x.
C. The bill stays flat because batch and on-demand average out.
D. The bill depends only on token count, not on model.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: "batch is cheaper" is true within a model tier, not across tiers; the bigger model dominates.
- B is correct: GPT-4o input is $1.25 vs mini's $0.15, so even at the 50% batch discount the new bill is roughly 4x the old one.
- C is wrong: batch discounts are 50%, not 87.5%.
- D is wrong: per-1M rates differ by an order of magnitude across model tiers; model choice is the largest single lever on the bill.

</details>